In [1]:
!pip install librecommender

In [1]:
import pandas as pd
from libreco.data import DatasetPure
from libreco.algorithms import UserCF

Instructions for updating:
non-resource variables are not supported in the long term


In [5]:
# train_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ratings_traindata_movielens.csv')
# val_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ratings_valdata_movielens.csv')
# test_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ratings_testdata_movielens.csv')
train_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_train_movielens.csv')
val_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_val_movielens.csv')
test_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_test_movielens.csv')
movies = pd.read_csv('processed_dataset/MovieLens-1M/movies/movies_movielens.csv')


In [6]:
items = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/books_data_with_new_id.csv')
# movies = pd.read_csv('processed_dataset/MovieLens-1M/movies/movies_movielens.csv')

train_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/binarized/amazon_books_ratings_train_fixed_binarized.csv')
val_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/binarized/amazon_books_ratings_val_fixed_binarized.csv')
test_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/binarized/amazon_books_ratings_test_fixed_binarized.csv')

# train_ratings = pd.read_csv('./processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_train.csv')
# val_ratings = pd.read_csv('./processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_val.csv')
# test_ratings = pd.read_csv('./processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_test.csv')

# movies = pd.read_csv('./processed_dataset/Amazon-Beauty/amazon_beauty_reviews_meta_modified.csv')


In [9]:
# Concatenate the two datasets
combined_ratings = pd.concat([train_ratings, val_ratings])

# Sort the combined dataset by the 'timestamp' column in ascending order
sorted_ratings = combined_ratings.sort_values(by='timestamp', ascending=True)

In [11]:
train_ratings = combined_ratings

In [12]:
train_ratings

,user_id,item_id,rating,timestamp
0,2.0,3931.0,7.0,1.957124e+09
1,2.0,1617.0,10.0,1.957124e+09
2,2.0,2309.0,8.0,1.957124e+09
3,2.0,1271.0,10.0,1.957124e+09
4,2.0,5027.0,6.0,1.957124e+09
...,...,...,...,...
99687,6040.0,3083.0,4.0,9.632721e+08
99688,6040.0,2366.0,3.0,9.632722e+08
99689,6040.0,3819.0,5.0,9.632722e+08
99690,6040.0,1900.0,5.0,9.648284e+08


In [13]:
# Rename columns to match the expected format
train_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
val_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
test_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)

# train_ratings.rename(columns={'user_id': 'user', 'parent_asin': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
# val_ratings.rename(columns={'user_id': 'user', 'parent_asin': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
# test_ratings.rename(columns={'user_id': 'user', 'parent_asin': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)

# Ensure the columns are in the correct order
train_ratings = train_ratings[['user', 'item', 'label', 'time']]
val_ratings = val_ratings[['user', 'item', 'label', 'time']]
test_ratings = test_ratings[['user', 'item', 'label', 'time']]

In [14]:
train_data, data_info = DatasetPure.build_trainset(train_ratings)
eval_data = DatasetPure.build_evalset(val_ratings)
test_data = DatasetPure.build_testset(test_ratings)
print(data_info)  # n_users: 5894, n_items: 3253, data sparsity: 0.4172 %

n_users: 6838, n_items: 7619, data density: 1.7226 %


In [15]:
train_ratings = train_ratings.sort_values(by='time')
val_ratings = val_ratings.sort_values(by='time')
test_ratings = test_ratings.sort_values(by='time')

In [11]:
test_ratings

,user,item,label,time
0,AZUNT3QP2CWTL,B000G167FA,1,-1
1,A3VVDE8I22IAJA,B0006D9LII,1,868492800
2,A3AZ4O4I9S4668,B000NKUH32,1,869270400
3,A319KYEIAZ3SON,B000Q1RMUE,1,869702400
4,A3SOB0CMUBK6XJ,B0006BV6RY,1,871084800
...,...,...,...,...
36021,A67VT05EV5EOF,B0006D1ZKU,1,1362009600
36023,A67VT05EV5EOF,B000CPSU9G,1,1362009600
36025,A2GDT5QQSFZD14,1593355548,1,1362268800
36024,A2GDT5QQSFZD14,1844560333,1,1362268800


In [16]:
user_cf = UserCF(task="ranking", data_info=data_info, k_sim=100, sim_type="cosine", mode='invert')

In [17]:
# Training the model
user_cf.fit(train_data, verbose=2, eval_data=eval_data, k=5, metrics=["loss", "roc_auc", "precision", "recall", "ndcg"], neg_sampling=True)

Training start time: 2024-09-07 12:35:53
Final block size and num: (6838, 1)
sim_matrix elapsed: 0.526s
sim_matrix, shape: (6838, 6838), num_elements: 7655434, density: 16.3724 %


eval_pointwise:   0%|          | 0/25 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 0 and item 4871, proceed with default prediction
No common interaction or similar neighbor for user 0 and item 190, proceed with default prediction
No common interaction or similar neighbor for user 0 and item 1700, proceed with default prediction
No common interaction or similar neighbor for user 0 and item 5611, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 5155, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 6797, proceed with default prediction


eval_listwise: 100%|██████████| 6040/6040 [00:30<00:00, 195.08it/s]

	 eval log_loss: 3.2245
	 eval roc_auc: 0.8379
	 eval precision@5: 0.0000
	 eval recall@5: 0.0000
	 eval ndcg@5: 0.0000


In [18]:
from libreco.evaluation import evaluate

# Evaluate the model on the test data with the specified metrics
evaluation_results = evaluate(
    model=user_cf,
    data=test_data,
    neg_sampling=True,
    metrics=["loss", "roc_auc", "precision", "recall", "ndcg"]
)

# Print the evaluation results
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

eval_pointwise:  19%|█▉        | 5/26 [00:01<00:05,  4.17it/s]

Detect 1 unknown interaction(s), position: [7262]


eval_pointwise:  31%|███       | 8/26 [00:01<00:04,  4.06it/s]

Detect 1 unknown interaction(s), position: [6664]


eval_pointwise:  58%|█████▊    | 15/26 [00:03<00:02,  4.14it/s]

...

Detect 2 unknown interaction(s), position: [5656, 6658]


eval_pointwise:  92%|█████████▏| 24/26 [00:05<00:00,  4.16it/s]

Detect 1 unknown interaction(s), position: [6880]


eval_listwise: 100%|██████████| 6040/6040 [00:30<00:00, 195.03it/s]


loss: 5.734623202335814
roc_auc: 0.7327538032559902
precision: 0.042748344370860926
recall: 0.035922650886634204
ndcg: 0.15222396329625607


In [19]:
unique_users = test_ratings['user'].unique()
print(len(unique_users))

6040


In [21]:
import numpy as np
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_user_cf_model(model, test_data, train_data, all_items, k):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_data['user'].unique()

    for user in unique_users:
        # Get items the user has seen in the training data
        seen_items = train_data[train_data['user'] == user]['item'].values

        # Get items the user has not seen
        # unseen_items = np.setdiff1d(all_items, seen_items)

        # Recommend items for the user using the model|
        recommended_items = model.recommend_user(user, n_rec=k, filter_consumed=True)
        recommended_items = recommended_items[user]
        user_test_data = test_data[test_data['user'] == user]
        test_items = user_test_data['item'].values

        y_score = [1 if item in test_items else 0 for item in recommended_items]
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))

    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }

all_items = movies['item_id'].unique()
# all_items = items['item_id'].unique()

# Evaluate the model
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_19140\4094785750.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.6582327871767616, 'Recall@5': 0.021671015806214857, 'MRR@5': 0.10364790286975717}
{'NDCG@10': 0.5417989029519072, 'Recall@10': 0.035922650886634204, 'MRR@10': 0.11594114632608009}


In [ ]:
{'NDCG@5': 0.7494054887964802, 'Recall@5': 0.06428771155699957, 'MRR@5': 0.15456895642080826}
{'NDCG@10': 0.6936378254664127, 'Recall@10': 0.07446002238290779, 'MRR@10': 0.15927973489701885}

In [17]:
def evaluate_user_cf_model(model, test_data, train_data, all_items, k):
    ndcg_scores = []
    recall_scores = []

    # Get unique users
    unique_users = test_data['user'].unique()

    for user in unique_users:
        # Get items the user has seen in the training data
        seen_items = train_data[train_data['user'] == user]['item'].values

        # Recommend items for the user using the model|
        recommended_items = model.recommend_user(user, n_rec=k, filter_consumed=True)
        recommended_items = recommended_items[user]
        user_test_data = test_data[test_data['user'] == user]
        test_items = user_test_data['item'].values
        labels = user_test_data['label'].values


        y_score = [
            user_test_data[user_test_data['item'] == item]['label'].values[0] if item in test_items else 0
            for item in recommended_items
        ]

        # y_score = [
        #     user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 2.5
        #     for item in recommended_items
        # ]

        ndcg = ndcg_at_k(y_score, k)
        # recall = recall_at_k(labels, k)

        ndcg_scores.append(ndcg)
        # recall_scores.append(recall)
        # print(user)
    # avg_ndcg = np.nanmean(ndcg_scores)
    # avg_recall = np.nanmean(recall_scores)
    avg_ndcg = np.nanmean(ndcg_scores)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    # avg_recall = np.mean(np.nan_to_num(recall_scores, nan=0.0))
    return {
        'NDCG@{}'.format(k): avg_ndcg,
        # 'Recall@{}'.format(k): avg_recall
    }

# all_items = movies['item_id'].unique()
all_items = items['item_id'].unique()

# Evaluate the model
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_9344\611941998.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.726869975750975}
{'NDCG@10': 0.6625759494461073}


In [ ]:
{'NDCG@5': 0.7482504128365846}
{'NDCG@10': 0.6943043653737456}

In [18]:
import numpy as np
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def evaluate_user_cf_model2(model, test_data, train_data, all_items, k):
    ndcg_scores = []
    recall_scores = []

    # Get unique users
    unique_users = test_data['user'].unique()

    for user in unique_users:
        # Get items the user has seen in the training data
        seen_items = train_data[train_data['user'] == user]['item'].values

        # Recommend items for the user using the model|
        recommended_items = model.recommend_user(user, n_rec=k, filter_consumed=True)
        recommended_items = recommended_items[user]
        user_test_data = test_data[test_data['user'] == user]
        test_items = user_test_data['item'].values
        labels = user_test_data['label'].values


        # y_score = [
        #     user_test_data[user_test_data['item'] == item]['label'].values[0] if item in test_items else 0
        #     for item in recommended_items
        # ]

        y_score = [
            user_test_data[user_test_data['item'] == item]['label'].values[0] if item in test_items else 2.5
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        # recall = recall_at_k(labels, k)

        ndcg_scores.append(ndcg)
        # recall_scores.append(recall)
        # print(user)
    # avg_ndcg = np.nanmean(ndcg_scores)
    # avg_recall = np.nanmean(recall_scores)
    avg_ndcg = np.nanmean(ndcg_scores)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    # avg_recall = np.mean(np.nan_to_num(recall_scores, nan=0.0))
    return {
        'NDCG@{}'.format(k): avg_ndcg,
        # 'Recall@{}'.format(k): avg_recall
    }

# all_items = movies['item_id'].unique()
all_items = items['item_id'].unique()

# Evaluate the model
eval_result = evaluate_user_cf_model2(user_cf, test_ratings, train_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model2(user_cf, test_ratings, train_ratings, all_items, k=10)
print(eval_result)

{'NDCG@5': 0.9853986791640111}
{'NDCG@10': 0.9855440278647153}


In [ ]:
{'NDCG@5': 0.986232785657715}
{'NDCG@10': 0.9843870995527803}

In [22]:
def evaluate_user_cf_model2(model, test_data, train_data, all_items, k):
    ndcg_scores = []
    recall_scores = []

    # Get unique users
    unique_users = test_data['user'].unique()

    for user in unique_users:
        # Get items the user has seen in the training data
        seen_items = train_data[train_data['user'] == user]['item'].values

        # Get items the user has not seen
        # unseen_items = np.setdiff1d(all_items, seen_items)

        # Recommend items for the user using the model|
        recommended_items = model.recommend_user(user, n_rec=k, filter_consumed=True)
        recommended_items = recommended_items[user]
        user_test_data = test_data[test_data['user'] == user]
        test_items = user_test_data['item'].values
        # labels = user_test_data['label'].values


        # y_score = [1 if item in test_items else 0 for item in recommended_items]
        y_score = [
            1 if (item in test_items and user_test_data[user_test_data['item'] == item]['label'].values[0] == 1) else 0
            for item in recommended_items
        ]
        ndcg = ndcg_at_k(y_score, k)
        # recall = recall_at_k(labels, k)

        ndcg_scores.append(ndcg)
        # recall_scores.append(recall)
        # print(user)
    # avg_ndcg = np.nanmean(ndcg_scores)
    # avg_recall = np.nanmean(recall_scores)
    avg_ndcg = np.nanmean(ndcg_scores)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    # avg_recall = np.mean(np.nan_to_num(recall_scores, nan=0.0))
    return {
        'NDCG@{}'.format(k): avg_ndcg,
        # 'Recall@{}'.format(k): avg_recall
    }

# all_items = movies['item_id'].unique()
all_items = items['item_id'].unique()

# Evaluate the model
eval_result = evaluate_user_cf_model2(user_cf, test_ratings, train_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model2(user_cf, test_ratings, train_ratings, all_items, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_9344\1262673704.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.7262661955054526}
{'NDCG@10': 0.661814837016558}
